# ライブラリのインポート

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import random
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score

# 変数の設定

In [ ]:
class CFG:
    VER = 1
    AUTHOR = "suba"
    COMPETITION = "atmacup17"
    DATA_PATH = Path("/workspace/kaggle_llm_book/ch3/data/takaito/atmacup17")
    SEED = 42
    TARGET_COL = "Recommended IND"
    TARGET_COL_CLASS_NUM = 2
    METRIC = "auc"
    METRIC_MAXIMIZE_FLAG = True

# 乱数の設定

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
seed_everything(CFG.SEED)

# データの読み込み

In [ ]:
clothing_master_df = pd.read_csv(CFG.DATA_PATH / "clothing_master.csv")
train_df = pd.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")

In [ ]:
# 訓練セット
train_deberta_df = pd.read_csv("/kaggle/input/kaggle-llm-book-3-4-bert-2025-09/takaito_deberta-v3-large_seed42_ver1_train_preds.csv")
train_gemma_df = pd.read_csv("/kaggle/input/kaggle-llm-book-3-5-2025-09/takaito_gemma-2-2b-it-bnb-4bit_seed7_ver1_train_preds.csv")
# テストセット
test_deberta_df = pd.read_csv("/kaggle/input/kaggle-llm-book-3-4-bert-2025-09/takaito_deberta-v3-large_seed42_ver1_test_preds.csv")
test_gemma_df = pd.read_csv("/kaggle/input/kaggle-llm-book-3-5-2025-09/takaito_gemma-2-2b-it-bnb-4bit_seed7_ver1_test_preds.csv")

In [ ]:
train_df["deberta"] = train_deberta_df["takaito_deberta-v3-large_seed42_ver1_pred_prob"]
train_df["gemma"] = train_gemma_df["takaito_gemma-2-2b-it-bnb-4bit_seed7_ver1_pred_prob"]
test_df["deberta"] = test_deberta_df["takaito_deberta-v3-large_seed42_ver1_pred_prob"]
test_df["gemma"] = test_gemma_df["takaito_gemma-2-2b-it-bnb-4bit_seed7_ver1_pred_prob"]

# 最適な重みの探索

In [ ]:
weight_list = []
score_list = []
for i in range(0, 101):
    weight = i / 100
    ensemble_preds = weight * train_df["gemma"] + (1 - weight) * train_df["deberta"]
    score = roc_auc_score(train_df[CFG.TARGET_COL], ensemble_preds)
    weight_list.append(weight)
    score_list.append(score)

In [ ]:
vis_df = pd.DataFrame()
vis_df["gemma_weight"] = weight_list
vis_df["roc_auc_score"] = score_list
vis_df.plot.scatter(x="gemma_weight", y="roc_auc_score")

In [ ]:
vis_df.sort_values("roc_auc_score").tail(3)

# 提出用のファイル作成

In [ ]:
weight = 0.69
ensemble_preds = weight * train_df["gemma"] + (1 - weight) * train_df["deberta"]
score = roc_auc_score(train_df[CFG.TARGET_COL], ensemble_preds)
print(score)
ensemble_preds = weight * test_df["gemma"] + (1 - weight) * test_df["deberta"]
test_df["target"] = ensemble_preds
test_df[["target"]].to_csv("ensemble_submission1.csv", index=False)

In [ ]:
weight = 0.90
ensemble_preds = weight * train_df["gemma"] + (1 - weight) * train_df["deberta"]
score = roc_auc_score(train_df[CFG.TARGET_COL], ensemble_preds)
print(score)
ensemble_preds = weight * test_df["gemma"] + (1 - weight) * test_df["deberta"]
test_df["target"] = ensemble_preds
test_df[["target"]].to_csv("ensemble_submission2.csv", index=False)